In [3]:

from collections import Counter
import re

EOW = "_"

def clean_words(text: str):
    text = re.sub(r"[^a-zA-Z\s]", " ", text).lower()   # remove punctuation
    return [w for w in text.split() if w]

def tok(w): return list(w) + [EOW]

def count_bigrams(words):
    c = Counter()
    for w in words: c.update(zip(w, w[1:]))
    return c

def merge_seq(seq, pair):
    a, b = pair
    out, i = [], 0
    while i < len(seq):
        if i < len(seq)-1 and seq[i] == a and seq[i+1] == b:
            out.append(a+b); i += 2
        else:
            out.append(seq[i]); i += 1
    return out

def learn_bpe(paragraph, n_merges=30):
    corpus = [tok(w) for w in clean_words(paragraph)]
    merges = []
    for s in range(1, n_merges+1):
        bg = count_bigrams(corpus)
        if not bg:
            print(f"Step {s:02d}: No bigrams left to merge."); break
        pair, cnt = bg.most_common(1)[0]
        corpus = [merge_seq(w, pair) for w in corpus]
        merges.append((pair, cnt))
        vocab = {t for w in corpus for t in w}
        print(f"Step {s:02d}: top pair = {pair} (count={cnt}) -> vocab size = {len(vocab)}")
    return merges, corpus, Counter(clean_words(paragraph))

def segment(word, merges):
    seq = tok(re.sub(r"[^a-zA-Z]", "", word).lower())
    for (pair, _cnt) in merges:
        seq = merge_seq(seq, pair)
    return seq

if __name__ == "__main__":
    paragraph = (
        "Natural language processing helps computers understand human language. "
        "Subword tokenization is useful for handling rare and unseen words. "
        "Models learn patterns from text to improve understanding. "
        "Effective tokenization reduces vocabulary size and improves performance. "
        "This approach allows systems to generalize better to new word forms."
    )

    merges, final_corpus, wf = learn_bpe(paragraph, n_merges=30)

    print("\n=== Top 5 merges ===")
    for i, (pair, cnt) in enumerate(merges[:5], 1):
        print(f"{i}. {pair} -> {pair[0]+pair[1]} (count={cnt})")

    print("\n=== 5 longest tokens ===")
    toks = sorted({t for w in final_corpus for t in w}, key=lambda x: (len(x), x), reverse=True)[:5]
    for t in toks:
        print(f"- {t} (len={len(t)})")

    # Choose 5 words (one rare + one derived/inflected). Edit if needed.
    test_words = ["language", "processing", "tokenization", "generalize", "vocabulary"]

    print("\n=== Segmentation of 5 words ===")
    for w in test_words:
        print(f"{w:<15} -> {' '.join(segment(w, merges))}")

    rare = sorted([w for w, c in wf.items() if c == 1])
    print("\nRare words (occur once):")
    print(", ".join(rare) if rare else "None found")


Step 01: top pair = ('s', '_') (count=12) -> vocab size = 26
Step 02: top pair = ('a', 'n') (count=9) -> vocab size = 27
Step 03: top pair = ('e', '_') (count=8) -> vocab size = 28
Step 04: top pair = ('e', 'r') (count=7) -> vocab size = 29
Step 05: top pair = ('o', 'r') (count=6) -> vocab size = 30
Step 06: top pair = ('r', 'o') (count=5) -> vocab size = 31
Step 07: top pair = ('an', 'd') (count=5) -> vocab size = 32
Step 08: top pair = ('t', 'o') (count=5) -> vocab size = 33
Step 09: top pair = ('a', 't') (count=4) -> vocab size = 34
Step 10: top pair = ('p', 'ro') (count=4) -> vocab size = 35
Step 11: top pair = ('e', 'n') (count=4) -> vocab size = 36
Step 12: top pair = ('i', 'z') (count=4) -> vocab size = 36
Step 13: top pair = ('a', 'l') (count=3) -> vocab size = 37
Step 14: top pair = ('i', 'n') (count=3) -> vocab size = 38
Step 15: top pair = ('in', 'g') (count=3) -> vocab size = 38
Step 16: top pair = ('ing', '_') (count=3) -> vocab size = 38
Step 17: top pair = ('t', 'er') (c